# Packet P-048 — Kitchen Sink Temporal Stability

**Decision question:** P-035 showed the original model has temporal drift (tau-b 0.05–0.14 train-old/test-new). P-046 found the kitchen sink model improves random CV by +0.040. Does the expanded feature set also improve temporal generalization?

**Fastest falsifier:** Run the same temporal split protocol from P-035 with the 31-feature model. If temporal tau-b doesn't improve, the extra features only help within-distribution.

**Success:** Kitchen sink temporal tau-b ≥ 0.20 (up from 0.05–0.14 baseline).
**Kill:** Kitchen sink temporal tau-b still < 0.10 — extra features don't help temporal generalization.

In [1]:
"""Cell 1 — Temporal split: train on old Ref_IDs, test on new."""
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from scipy.stats import kendalltau
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('perovskite_stability_clean.csv')

ORIGINAL_16 = [
    'Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
    'MA', 'FA', 'Cs',
    'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
    'Perovskite_thickness', 'Cell_area_measured',
    'JV_default_Voc', 'JV_default_Jsc', 'JV_default_FF'
]

KITCHEN_SINK = ORIGINAL_16 + [
    'DMF', 'DMSO', 'other_solvent', 'DMF_DMSO_ratio',
    'LLE_1', 'LLE_2', 'LLE_3', 'LLE_4',
    'Perovskite_annealing_thermal_exposure',
    'Backcontact_thickness_list', 'ETL_thickness', 'HTL_thickness_list',
    'others_A', 'others_B', 'others_X'
]

TARGET = 'Stability_PCE_T80'
ET_PARAMS = dict(n_estimators=700, max_features='sqrt', min_samples_split=3,
                 min_samples_leaf=1, bootstrap=False, random_state=42)

y = np.log1p(df[TARGET])

# Use Ref_ID as temporal proxy (lower Ref_ID = older publication)
ref_ids = df['Ref_ID'].values
unique_refs = np.sort(df['Ref_ID'].unique())

# Multiple temporal split points (60/40, 70/30, 80/20)
split_ratios = [0.6, 0.7, 0.8]

print(f"{'=' * 80}")
print("TEMPORAL SPLIT — ORIGINAL 16 vs KITCHEN SINK 31")
print(f"{'=' * 80}")
print(f"Total devices: {len(df)}, Unique Ref_IDs: {len(unique_refs)}")

results = []

for split_pct in split_ratios:
    cutoff_idx = int(len(unique_refs) * split_pct)
    train_refs = set(unique_refs[:cutoff_idx])
    test_refs = set(unique_refs[cutoff_idx:])
    
    train_mask = df['Ref_ID'].isin(train_refs)
    test_mask = df['Ref_ID'].isin(test_refs)
    
    n_train = train_mask.sum()
    n_test = test_mask.sum()
    
    print(f"\n--- Split {split_pct:.0%} train / {1-split_pct:.0%} test ---")
    print(f"Train: {n_train} devices ({len(train_refs)} refs), Test: {n_test} devices ({len(test_refs)} refs)")
    
    for name, feats in [('Original 16', ORIGINAL_16), ('Kitchen Sink 31', KITCHEN_SINK)]:
        X = df[feats].fillna(0)
        
        X_tr, X_te = X[train_mask], X[test_mask]
        y_tr, y_te = y[train_mask], y[test_mask]
        
        model = ExtraTreesRegressor(**ET_PARAMS)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_te)
        
        tau, p = kendalltau(y_te, preds)
        
        results.append({
            'split': f"{split_pct:.0%}/{1-split_pct:.0%}",
            'model': name,
            'n_train': n_train,
            'n_test': n_test,
            'tau_b': tau,
            'p_value': p,
        })
        
        print(f"  {name:<25} tau-b: {tau:.4f}  (p={p:.4e})")

# Summary comparison
print(f"\n{'=' * 80}")
print("SUMMARY")
print(f"{'=' * 80}")
print(f"{'Split':<15} {'Original 16':>15} {'Kitchen Sink':>15} {'Delta':>10}")
print("-" * 58)

res_df = pd.DataFrame(results)
for split in res_df['split'].unique():
    orig = res_df[(res_df['split'] == split) & (res_df['model'] == 'Original 16')]['tau_b'].values[0]
    ks = res_df[(res_df['split'] == split) & (res_df['model'] == 'Kitchen Sink 31')]['tau_b'].values[0]
    print(f"{split:<15} {orig:>15.4f} {ks:>15.4f} {ks-orig:>+10.4f}")

mean_orig = res_df[res_df['model'] == 'Original 16']['tau_b'].mean()
mean_ks = res_df[res_df['model'] == 'Kitchen Sink 31']['tau_b'].mean()
print(f"{'Mean':<15} {mean_orig:>15.4f} {mean_ks:>15.4f} {mean_ks-mean_orig:>+10.4f}")

TEMPORAL SPLIT — ORIGINAL 16 vs KITCHEN SINK 31
Total devices: 1543, Unique Ref_IDs: 1543

--- Split 60% train / 40% test ---
Train: 925 devices (925 refs), Test: 618 devices (618 refs)


  Original 16               tau-b: 0.1073  (p=7.0501e-05)


  Kitchen Sink 31           tau-b: 0.1192  (p=1.0046e-05)

--- Split 70% train / 30% test ---
Train: 1080 devices (1080 refs), Test: 463 devices (463 refs)


  Original 16               tau-b: 0.0905  (p=3.7499e-03)


  Kitchen Sink 31           tau-b: 0.1178  (p=1.6220e-04)

--- Split 80% train / 20% test ---
Train: 1234 devices (1234 refs), Test: 309 devices (309 refs)


  Original 16               tau-b: 0.1459  (p=1.4231e-04)


  Kitchen Sink 31           tau-b: 0.1541  (p=5.8435e-05)

SUMMARY
Split               Original 16    Kitchen Sink      Delta
----------------------------------------------------------
60%/40%                  0.1073          0.1192    +0.0119
70%/30%                  0.0905          0.1178    +0.0273
80%/20%                  0.1459          0.1541    +0.0082
Mean                     0.1146          0.1304    +0.0158


In [2]:
"""Cell 2 — Evaluate and save."""

mean_ks_temporal = res_df[res_df['model'] == 'Kitchen Sink 31']['tau_b'].mean()
mean_orig_temporal = res_df[res_df['model'] == 'Original 16']['tau_b'].mean()

if mean_ks_temporal >= 0.20:
    status = "Confirmed"
elif mean_ks_temporal < 0.10:
    status = "Negative"
else:
    status = "Promising"

res_df.to_csv('Packet_P048_Kitchen_Sink_Temporal.csv', index=False)
print(f"Saved: Packet_P048_Kitchen_Sink_Temporal.csv")

print(f"\n{'=' * 60}")
print(f"P-048 STATUS: {status}")
print(f"{'=' * 60}")
print(f"Mean temporal tau-b (Original 16):   {mean_orig_temporal:.4f}")
print(f"Mean temporal tau-b (Kitchen Sink):  {mean_ks_temporal:.4f}")
print(f"Improvement:                         {mean_ks_temporal - mean_orig_temporal:+.4f}")

if status == "Confirmed":
    print("\nKitchen sink features rescue temporal generalization!")
elif status == "Negative":
    print("\nExtra features don't help temporal generalization.")
    print("The temporal drift is not about missing features.")
else:
    print("\nSome improvement but still below 0.20 threshold.")
    print("Fabrication features partially reduce temporal drift.")

Saved: Packet_P048_Kitchen_Sink_Temporal.csv

P-048 STATUS: Promising
Mean temporal tau-b (Original 16):   0.1146
Mean temporal tau-b (Kitchen Sink):  0.1304
Improvement:                         +0.0158

Some improvement but still below 0.20 threshold.
Fabrication features partially reduce temporal drift.
